## Restructure Attention Shift to have trials, sub-blocks and blocks

This script works with files of the form `_events_temp2.tsv` with columns
`onset`, `duration`, `sample`, `trial_type`, `value`, `event_code`, and `cond_code`.
The `key` is of the form `sub-xxx_run-y` which
uniquely specify each event file in the dataset.

**Transformations:**
1. Delete the `trial_type` and the `value` column.
2. Rename `repetition_type` as `rep_status` and `trigger` as `value`.
3. Insert a column called `trial` with the trial number. (Trial anchors are `show_face_initial`
and `show_cross`.  The exclude tags are `setup_left_sym` and `setup_right_sym`.
4. The `show_cross` value column should be 1.
5. Insert new column `rep_lag`.
6. Reorder the columns as `onset`, `duration`, `sample`, `event_type`, `face_type`,
`rep_status`, `rep_lag`, `value`, and `stim_file`.
7. Save as `*_events_temp2.tsv`

In [3]:
from hed.tools.io_utils import get_file_list, make_file_dict
from hed.tools.data_utils import get_new_dataframe
bids_root_path = 'G:/AttentionShift/AttentionShiftExperiments'
name_suffix = "_events_temp2"
name_indices = (0, -2)
files_bids = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix=name_suffix)
dict_bids = make_file_dict(files_bids, indices=name_indices)
print(f"\n{len(list(dict_bids))} Attention shift event files")


49 Attention shift event files


In [4]:
for key, file in dict_bids.items():
    df = get_new_dataframe(file)

    ## Set the sub blocks
    sub_mask = df['event_code'].map(str).isin(['1', '2']).astype(int)
    sub_total = 0
    sub_list = list(sub_mask)
    for i, value in sub_mask.iteritems():
        if value:
            sub_total += sub_list[i]
        sub_list[i] = sub_total
    df['sub_block'] = sub_list
    pause_col = df['event_code'].map(str) == '202'
    df.loc[pause_col, 'sub_block'] = 'n/a'

    ## Set the trials
    stim_total = 0
    stim_mask = \
        df['event_code'].map(str).isin(['3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']).astype(int)
    stim_list = list(stim_mask)
    for i, value in stim_list.iteritems():
        if value:
            stim_total += stim_list[i]
        stim_mask.iloc[i] = stim_total
    df['trial'] = sub_list
    df.loc[pause_col, 'trial'] = 'n/a'
    print("to here")
       # df_new.to_csv(eeg_corrected, sep='\t', index=False)
    break

to here
